In [1]:
import pandas as pd
from pathlib import Path
import re

## 1. Import Required Libraries
### Step Purpose: Setup & Validation
We import pandas for data manipulation, pathlib for file handling, and regex for text processing - essential tools for clinical data cleaning.

**Why This Process/Method Was Needed:**
- **Reasoning**: Clinical data analysis requires specialized libraries for handling large datasets, file operations, and text manipulation
- **How It Helps Analysis**: Enables efficient data loading, transformation, and processing at scale
- **Analysis Impact If Skipped**: Without these libraries, no data manipulation is possible - the entire analysis pipeline fails

# Clinical Data Cleaning and Analysis Pipeline

This notebook processes clinical CSV files, cleans them, detects outliers, and generates summary statistics.

---

## Why Clinical Data Cleaning Matters

**The Problem with Raw Clinical Data:**
- Mixed formats and inconsistent column naming
- Extra whitespace and encoding issues
- Missing values and empty rows/columns
- Measurement errors and sensor malfunctions (outliers)
- Text-encoded numbers preventing calculations

**The Impact of Skipping Cleaning:**
- ❌ Statistical bias (wrong means, correlations, predictions)
- ❌ Code breaks on unexpected data formats
- ❌ Machine learning models learn from garbage data
- ❌ Impossible to replicate results later
- ❌ Clinical decisions made on corrupted data

**This Pipeline Solves These Problems:**

| Cleaning Step | Raw Data Problem | Solution Applied | Analysis Impact if Skipped |
|---|---|---|---|
| **Column Name Standardization** | "Heart Rate", "HR", "heart_rate" | Convert all to lowercase_underscore | Code breaks, inconsistent queries |
| **Text Trimming** | Whitespace causes duplicate values | Strip all leading/trailing spaces | Grouping/aggregation fails silently |
| **Type Conversion** | Numbers stored as text strings | Auto-detect and convert numeric columns | Math operations impossible on text |
| **Empty Removal** | Wasted memory on all-null rows/cols | Drop entirely empty rows and columns | Misleading row counts, memory waste |
| **Numeric Rounding** | Floating-point precision errors | Round to 3 decimals | Statistics have unnecessary noise |
| **Outlier Detection** | Errors, sensor failures, false extremes | IQR method identifies >99.3% bounds | One erroneous value skews entire analysis |

**Result:** Clean, consistent, analysis-ready data enabling trustworthy insights.

---

In [2]:
file_names = [
    "HUPA0024P.csv",
    "HUPA0025P.csv",
    "HUPA0026P.csv",
    "HUPA0027P.csv",
    "HUPA0028P.csv"
]

for file in file_names:
    print(file, Path(file).exists())

HUPA0024P.csv True
HUPA0025P.csv True
HUPA0026P.csv True
HUPA0027P.csv True
HUPA0028P.csv True


## 2. Check Input Files Availability
### Step Purpose: Data Validation
This verifies all required CSV files exist before starting the cleaning process.

**Why This Process/Method Was Needed:**
- **Reasoning**: Clinical data pipelines must verify data availability upfront to prevent mid-process failures
- **How It Helps Analysis**: Ensures all expected data sources are present before investing computational resources
- **Analysis Impact If Skipped**: The notebook would crash during processing, wasting time and potentially corrupting partial results

In [3]:
output_folder = Path("cleaned_data")
output_folder.mkdir(exist_ok=True)

## 3. Create Output Directory
### Step Purpose: Prepare Storage Infrastructure
Creates a dedicated folder for all output files.

**Why This Process/Method Was Needed:**
- **Reasoning**: Organizing cleaned data separately from raw data maintains data integrity and prevents accidental overwrites
- **How It Helps Analysis**: Easy tracking of pipeline outputs and enables reproducibility
- **Analysis Impact If Skipped**: Cleaned data gets mixed with raw data, making it difficult to identify which files are safe to use and creating confusion in analysis

In [4]:
def clean_column_name(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name

## 4. Define Column Name Cleaning Function
### Step Purpose: Standardize Column Names
Converts all column headers to consistent format (lowercase, underscores, no special characters).

**Cleaning Logic:**
1. Convert to lowercase → Ensures consistency
2. Remove special characters → Prevents SQL/Python syntax errors
3. Compress multiple underscores → Clean, readable format
4. Strip leading/trailing underscores → Professional formatting

**Why This Process/Method Was Needed:**
- **Reasoning**: Clinical datasets often have inconsistent headers (spaces, mixed case, special chars)
- **How It Helps Analysis**: Enables reliable column referencing in analysis code and prevents programmatic errors
- **Analysis Impact If Skipped**: Code breaks when referencing columns with spaces or special characters, making analysis brittle and inconsistent

In [5]:
def find_outliers_iqr(df):
    numeric_columns = df.select_dtypes(include="number").columns
    outlier_rows = []

    for column in numeric_columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

        outlier_rows.append({
            "column": column,
            "lower_bound": round(lower_bound, 3),
            "upper_bound": round(upper_bound, 3),
            "outlier_count": int(outlier_count)
        })

    if len(outlier_rows) == 0:
        return pd.DataFrame(columns=[
            "column",
            "lower_bound",
            "upper_bound",
            "outlier_count"
        ])

    outlier_df = pd.DataFrame(outlier_rows)

    return outlier_df.sort_values(
        by="outlier_count",
        ascending=False
    )

## 5. Define Outlier Detection Function (IQR Method)
### Step Purpose: Identify Anomalous Values
Uses the Interquartile Range (IQR) Method to detect statistical outliers in clinical measurements.

**How IQR Works:**
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1 (middle 50% of data)
- Outliers: Values beyond Q1 - 1.5×IQR or Q3 + 1.5×IQR
- This captures ~99.3% of normal data in a distribution

**Why This Process/Method Was Needed:**
- **Reasoning**: Clinical data contains measurement errors, device malfunctions, and extreme events - IQR is robust and doesn't assume normal distribution
- **How It Helps Analysis**: Detects data entry errors and flags equipment failures, enabling focused investigation
- **Analysis Impact If Skipped**: Erroneous values bias all statistical analyses, making clinical conclusions dangerously wrong

In [6]:
def clean_dataset(df):
    df = df.copy()

    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    df.columns = [clean_column_name(col) for col in df.columns]

    text_columns = df.select_dtypes(include="object").columns

    for column in text_columns:
        df[column] = df[column].astype("string").str.strip()

    for column in df.columns:
        converted = pd.to_numeric(df[column], errors="coerce")

        if converted.notna().sum() > 0:
            df[column] = converted

    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].round(3)

    outliers = find_outliers_iqr(df)

    return df, outliers

## 6. Define Data Cleaning Function
### Step Purpose: Comprehensive Data Cleaning
This function applies 6 sequential cleaning operations:

**Cleaning Step 1: Drop Completely Empty Rows/Columns**
- Removes rows where ALL values are missing
- Removes columns where ALL values are missing
- Why: Empty rows/columns provide no information and waste memory

**Cleaning Step 2: Standardize Column Names**
- Applies the column cleaning function to all headers
- Why: Ensures consistent naming across all datasets

**Cleaning Step 3: Clean Text Data**
- Strips leading/trailing whitespace from text columns
- Why: Extra spaces cause duplicate values and break grouping/matching

**Cleaning Step 4: Auto-Detect & Convert Numeric Columns**
- Attempts to convert all columns to numbers
- Only converts if >0 values successfully convert (not all text)
- Why: Some numeric data may be imported as text strings, preventing calculations

**Cleaning Step 5: Round Numeric Precision**
- Rounds all numbers to 3 decimal places
- Why: Removes floating-point errors and measurement noise

**Cleaning Step 6: Detect Outliers**
- Runs IQR analysis on all numeric columns
- Returns outlier report for investigation

**Why This Process/Method Was Needed:**
- **Reasoning**: Raw clinical data has multiple quality issues that must be addressed systematically
- **How It Helps Analysis**: Produces clean, consistent data ready for reliable statistical analysis
- **Analysis Impact If Skipped**: All downstream analysis becomes unreliable, statistics are biased, visualizations are misleading

In [15]:
cleaned_data = {}
outlier_reports = {}

for file_name in file_names:
    print("=" * 60)
    print("Processing:", file_name)

    df_raw = pd.read_csv(file_name, sep=";")

    print("Original shape:", df_raw.shape)

    df_clean, outliers = clean_dataset(df_raw)

    print("Cleaned shape:", df_clean.shape)

    cleaned_data[file_name] = df_clean
    outlier_reports[file_name] = outliers

    cleaned_file = file_name.replace(".csv", "_cleaned.csv")

    # Save ONLY cleaned file
    df_clean.to_csv(output_folder / cleaned_file, index=False)

    print("Saved:", cleaned_file)

print("DONE")

Processing: HUPA0024P.csv
Original shape: (2902, 8)
Cleaned shape: (2902, 8)
Saved: HUPA0024P_cleaned.csv
Processing: HUPA0025P.csv
Original shape: (4006, 8)
Cleaned shape: (4006, 8)
Saved: HUPA0025P_cleaned.csv
Processing: HUPA0026P.csv
Original shape: (40605, 8)
Cleaned shape: (40605, 8)
Saved: HUPA0026P_cleaned.csv
Processing: HUPA0027P.csv
Original shape: (165306, 8)
Cleaned shape: (165306, 8)
Saved: HUPA0027P_cleaned.csv
Processing: HUPA0028P.csv
Original shape: (25902, 8)
Cleaned shape: (25902, 8)
Saved: HUPA0028P_cleaned.csv
DONE


## 7. Process and Clean All Input Files
### Step Purpose: Batch Process All Files
Loops through each clinical data file, applies all cleaning operations, and saves results.

**Why This Process/Method Was Needed:**
- **Reasoning**: Batch processing ensures consistency across all 5 files and documents each step's impact
- **How It Helps Analysis**: Removes manual, error-prone per-file processing and creates permanent cleaned files for reproducibility
- **Analysis Impact If Skipped**: Different people might clean different files differently, making results inconsistent and unreplicable

In [10]:
for file_name, df in cleaned_data.items():
    print("=" * 60)
    print("Preview:", file_name)
    display(df.head())

Preview: HUPA0024P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-01-20T11:30:00,87.000,8.187,67.364,20.0,0.069,0.0,0.0
1,2020-01-20T11:35:00,86.000,17.988,75.250,107.0,0.069,0.0,0.0
2,2020-01-20T11:40:00,85.000,17.412,76.778,123.0,0.069,0.0,0.0
3,2020-01-20T11:45:00,84.000,10.378,83.667,30.0,0.069,0.0,0.0
4,2020-01-20T11:50:00,85.333,14.529,80.289,74.0,0.069,0.0,0.0


Preview: HUPA0025P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-01-16T13:45:00,108.000,17.583,80.829,54.0,0.0,0.0,0.0
1,2020-01-16T13:50:00,103.667,11.344,74.730,22.0,0.0,0.0,0.0
2,2020-01-16T13:55:00,99.333,13.187,76.444,14.0,0.0,0.0,0.0
3,2020-01-16T14:00:00,95.000,12.762,80.152,17.0,0.0,0.0,0.0
4,2020-01-16T14:05:00,98.333,15.598,74.952,71.0,0.0,0.0,0.0


Preview: HUPA0026P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-05-23T00:15:00,145.000,4.997,85.533,0.0,0.056,0.0,0.0
1,2020-05-23T00:20:00,144.667,5.251,86.226,0.0,0.056,0.0,0.0
2,2020-05-23T00:25:00,144.333,12.026,97.556,112.0,0.056,0.0,0.0
3,2020-05-23T00:30:00,144.000,12.619,98.698,69.0,0.056,0.0,0.0
4,2020-05-23T00:35:00,143.667,10.332,96.323,69.0,0.056,0.0,0.0


Preview: HUPA0027P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-06-26T22:15:00,112.0,22.655,81.000,261.0,0.066,0.0,0.0
1,2020-06-26T22:20:00,105.0,21.162,84.625,200.0,0.066,0.0,0.0
2,2020-06-26T22:25:00,98.0,25.643,89.222,292.0,0.066,0.0,0.0
3,2020-06-26T22:30:00,91.0,12.572,80.028,68.0,0.066,0.0,0.0
4,2020-06-26T22:35:00,91.0,6.722,75.108,0.0,0.066,0.0,0.0


Preview: HUPA0028P.csv


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2022-02-17T13:50:00,78.333,13.581,86.040,198.0,0.0,0.0,0.0
1,2022-02-17T13:55:00,77.667,28.421,121.809,374.0,0.0,0.0,0.0
2,2022-02-17T14:00:00,77.000,15.919,101.689,96.0,0.0,0.0,0.0
3,2022-02-17T14:05:00,76.667,8.814,91.309,0.0,0.0,0.0,0.0
4,2022-02-17T14:10:00,76.333,8.454,93.474,0.0,0.0,0.0,0.0


## 8. Display Preview of Cleaned Data
### Step Purpose: Verify Cleaned Data Quality
Displays first 5 rows of each cleaned dataset to visually confirm cleaning worked correctly.

**Why This Process/Method Was Needed:**
- **Reasoning**: Human eyes catch issues that automated checks miss (e.g., unexpected date formats)
- **How It Helps Analysis**: Quick sanity check before downstream analysis and catches unexpected transformations
- **Analysis Impact If Skipped**: You wouldn't notice if cleaning broke something until analysis gives wrong results

In [11]:
for file_name, report in outlier_reports.items():
    print("=" * 60)
    print("Outliers:", file_name)
    display(report)

Outliers: HUPA0024P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-138.000,230.000,150
5,bolus_volume_delivered,0.000,0.000,21
6,carb_input,0.000,0.000,14
1,calories,-15.334,40.933,5
0,glucose,-46.375,375.959,0
2,heart_rate,20.299,126.769,0
4,basal_rate,-0.104,0.173,0


Outliers: HUPA0025P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-31.500,52.500,674
1,calories,-2.055,22.332,399
5,bolus_volume_delivered,0.000,0.000,130
2,heart_rate,44.241,111.942,52
0,glucose,3.168,220.500,23
6,carb_input,0.000,0.000,20
4,basal_rate,-0.074,0.258,0


Outliers: HUPA0026P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,-81.000,135.000,5426
1,calories,-1.018,12.942,4820
5,bolus_volume_delivered,0.000,0.000,334
0,glucose,-40.500,360.832,212
2,heart_rate,43.951,115.282,95
6,carb_input,0.000,0.000,46
4,basal_rate,-0.084,0.140,0


Outliers: HUPA0027P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,0.000,0.000,38057
1,calories,1.169,14.465,22933
2,heart_rate,31.871,116.388,5528
0,glucose,3.356,250.786,2680
5,bolus_volume_delivered,0.000,0.000,1876
6,carb_input,0.000,0.000,1531
4,basal_rate,-0.099,0.165,0


Outliers: HUPA0028P.csv


,column,lower_bound,upper_bound,outlier_count
3,steps,0.000,0.000,5526
1,calories,2.204,8.320,3910
2,heart_rate,41.072,106.157,852
0,glucose,31.744,222.553,255
5,bolus_volume_delivered,0.000,0.000,223
6,carb_input,0.000,0.000,217
4,basal_rate,-0.063,0.105,0


## 9. Display Outlier Reports
### Step Purpose: Investigate Anomalies
Displays detailed outlier reports showing which columns have suspicious values.

**Report Includes:**
- Column name
- IQR bounds (lower/upper threshold)
- Count of outliers detected

**Why This Process/Method Was Needed:**
- **Reasoning**: Outliers could be real extreme events or errors (typo, sensor malfunction)
- **How It Helps Analysis**: Distinguishes good data from bad and informs decisions on exclusion or investigation
- **Analysis Impact If Skipped**: Outliers silently corrupt statistics, making clinical findings unreliable

In [12]:
summary = []

for file_name, df in cleaned_data.items():
    outliers = outlier_reports[file_name]

    summary.append({
        "file_name": file_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "numeric_columns": len(df.select_dtypes(include="number").columns),
        "total_outliers": outliers["outlier_count"].sum()
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

,file_name,rows,columns,numeric_columns,total_outliers
0,HUPA0024P.csv,2902,8,7,190
1,HUPA0025P.csv,4006,8,7,1298
2,HUPA0026P.csv,40605,8,7,10933
3,HUPA0027P.csv,165306,8,7,72605
4,HUPA0028P.csv,25902,8,7,10983


## 10. Generate Data Quality Summary
### Step Purpose: Generate Overall Data Quality Metrics
Creates summary table showing:
- Number of rows & columns per file
- Count of numeric columns
- Total outliers detected

**Why This Process/Method Was Needed:**
- **Reasoning**: High-level metrics help assess data volume and quality in one glance
- **How It Helps Analysis**: Quick comparison across all files and detects problematic files with more outliers
- **Analysis Impact If Skipped**: No easy way to compare data volumes and quality across files, harder to spot issues

In [13]:
clinical_summary = []

for file_name, df in cleaned_data.items():

    summary = {
        "file_name": file_name
    }

    # Glucose features
    if "glucose" in df.columns:
        summary["glucose_mean"] = round(df["glucose"].mean(), 3)
        summary["glucose_std"] = round(df["glucose"].std(), 3)
        summary["glucose_min"] = round(df["glucose"].min(), 3)
        summary["glucose_max"] = round(df["glucose"].max(), 3)

    # Heart rate features
    if "heart_rate" in df.columns:
        summary["hr_mean"] = round(df["heart_rate"].mean(), 3)
        summary["hr_std"] = round(df["heart_rate"].std(), 3)
        summary["hr_max"] = round(df["heart_rate"].max(), 3)

    # Steps
    if "steps" in df.columns:
        summary["steps_sum"] = round(df["steps"].sum(), 3)

    # Carbs
    if "carb_input" in df.columns:
        summary["carbs_sum"] = round(df["carb_input"].sum(), 3)

    # Bolus
    if "bolus_volume_delivered" in df.columns:
        summary["bolus_sum"] = round(df["bolus_volume_delivered"].sum(), 3)

    # Basal
    if "basal_rate" in df.columns:
        summary["basal_mean"] = round(df["basal_rate"].mean(), 3)

    clinical_summary.append(summary)

clinical_features_df = pd.DataFrame(clinical_summary)

display(clinical_features_df)

,file_name,glucose_mean,glucose_std,glucose_min,glucose_max,hr_mean,hr_std,hr_max,steps_sum,carbs_sum,bolus_sum,basal_mean
0,HUPA0024P.csv,166.944,66.546,42.0,359.0,73.794,15.898,117.597,163403.0,76.50,71.00,0.042
1,HUPA0025P.csv,113.846,40.053,40.0,245.0,79.110,13.710,195.615,111473.0,58.50,360.65,0.094
2,HUPA0026P.csv,162.985,68.665,40.0,422.0,80.285,12.016,156.909,1929063.0,176.20,2993.00,0.036
3,HUPA0027P.csv,130.830,46.267,40.0,397.0,76.785,16.567,178.353,4380913.0,5161.25,12841.10,0.040
4,HUPA0028P.csv,128.436,35.451,40.0,296.0,75.643,13.320,149.273,492727.0,711.50,778.00,0.017


## 11. Calculate Clinical Features Summary
### Step Purpose: Extract Clinical Insights
Calculates summary statistics for key clinical measurements:
- **Glucose**: Mean, Std Dev, Min, Max (shows glycemic control)
- **Heart Rate**: Mean, Std Dev, Max (cardiovascular stress)
- **Steps**: Sum (physical activity level)
- **Carbs**: Sum (dietary intake)
- **Bolus**: Sum (insulin delivery)
- **Basal**: Mean (baseline insulin rate)

**Why This Process/Method Was Needed:**
- **Reasoning**: Aggregated clinical metrics enable rapid understanding of patient state
- **How It Helps Analysis**: Single-row summary per file/patient enables trend analysis and comparison
- **Analysis Impact If Skipped**: Thousands of raw data points are overwhelming, can't easily spot patient differences

In [14]:
clinical_features_df.to_csv(
    output_folder / "clinical_features_summary.csv",
    index=False
)

print("Clinical feature summary saved.")

Clinical feature summary saved.


## 12. Save Clinical Features Summary to CSV
### Step Purpose: Persist Results
Saves clinical features summary to CSV for downstream use (reporting, dashboards, ML models).

**Why This Process/Method Was Needed:**
- **Reasoning**: Clinical workflows require persistent, reproducible outputs
- **How It Helps Analysis**: Results survive session exit and can be imported into other tools
- **Analysis Impact If Skipped**: Data only exists in notebook memory, can't reuse in other workflows

In [16]:
combined_df = pd.concat(cleaned_data.values(), ignore_index=True)

combined_df.to_csv(
    output_folder / "combined_cleaned_data.csv",
    index=False
)

print("Combined dataset created.")
print("Shape:", combined_df.shape)

display(combined_df.head())

Combined dataset created.
Shape: (238721, 8)


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2020-01-20T11:30:00,87.000,8.187,67.364,20.0,0.069,0.0,0.0
1,2020-01-20T11:35:00,86.000,17.988,75.250,107.0,0.069,0.0,0.0
2,2020-01-20T11:40:00,85.000,17.412,76.778,123.0,0.069,0.0,0.0
3,2020-01-20T11:45:00,84.000,10.378,83.667,30.0,0.069,0.0,0.0
4,2020-01-20T11:50:00,85.333,14.529,80.289,74.0,0.069,0.0,0.0


## 13. Combine All Cleaned Files
### Step Purpose: Create Combined Dataset
Concatenates all cleaned datasets into a single CSV for holistic analysis.

**Why This Process/Method Was Needed:**
- **Reasoning**: Enables cross-file analysis and pattern detection across all patient data
- **How It Helps Analysis**: Simplifies working with all data in one table for comprehensive insights
- **Analysis Impact If Skipped**: Can't perform dataset-wide analyses or model training on combined data

In [18]:
combined_dfs = []

for file_name, df in cleaned_data.items():

    temp_df = df.copy()

    # Extract patient id from filename
    patient_id = file_name.replace(".csv", "")

    # Add patient_id column
    temp_df["patient_id"] = patient_id

    combined_dfs.append(temp_df)

# Combine all datasets
combined_df = pd.concat(combined_dfs, ignore_index=True)

# Save combined dataset
combined_df.to_csv(
    output_folder / "combined_cleaned_data.csv",
    index=False
)

print("Combined dataset created successfully.")
print("Shape:", combined_df.shape)

display(combined_df.head())

Combined dataset created successfully.
Shape: (238721, 9)


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id
0,2020-01-20T11:30:00,87.000,8.187,67.364,20.0,0.069,0.0,0.0,HUPA0024P
1,2020-01-20T11:35:00,86.000,17.988,75.250,107.0,0.069,0.0,0.0,HUPA0024P
2,2020-01-20T11:40:00,85.000,17.412,76.778,123.0,0.069,0.0,0.0,HUPA0024P
3,2020-01-20T11:45:00,84.000,10.378,83.667,30.0,0.069,0.0,0.0,HUPA0024P
4,2020-01-20T11:50:00,85.333,14.529,80.289,74.0,0.069,0.0,0.0,HUPA0024P


## 14. Add Patient Identifier and Recombine
### Step Purpose: Tag Data with Source Patient
Adds patient_id column to each dataset before combining, ensuring source traceability.

**Why This Process/Method Was Needed:**
- **Reasoning**: When combining multiple patient files, need to track which data belongs to which patient
- **How It Helps Analysis**: Enables patient-level grouping and comparison in combined dataset
- **Analysis Impact If Skipped**: Loss of provenance - can't distinguish between patients in combined data

In [19]:
# Convert time column to UTC

for file_name, df in cleaned_data.items():

    if "time" in df.columns:

        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        )

        print(f"{file_name} converted to UTC")

        display(df[["time"]].head())

HUPA0024P.csv converted to UTC


,time
0,2020-01-20 11:30:00+00:00
1,2020-01-20 11:35:00+00:00
2,2020-01-20 11:40:00+00:00
3,2020-01-20 11:45:00+00:00
4,2020-01-20 11:50:00+00:00


HUPA0025P.csv converted to UTC


,time
0,2020-01-16 13:45:00+00:00
1,2020-01-16 13:50:00+00:00
2,2020-01-16 13:55:00+00:00
3,2020-01-16 14:00:00+00:00
4,2020-01-16 14:05:00+00:00


HUPA0026P.csv converted to UTC


,time
0,2020-05-23 00:15:00+00:00
1,2020-05-23 00:20:00+00:00
2,2020-05-23 00:25:00+00:00
3,2020-05-23 00:30:00+00:00
4,2020-05-23 00:35:00+00:00


HUPA0027P.csv converted to UTC


,time
0,2020-06-26 22:15:00+00:00
1,2020-06-26 22:20:00+00:00
2,2020-06-26 22:25:00+00:00
3,2020-06-26 22:30:00+00:00
4,2020-06-26 22:35:00+00:00


HUPA0028P.csv converted to UTC


,time
0,2022-02-17 13:50:00+00:00
1,2022-02-17 13:55:00+00:00
2,2022-02-17 14:00:00+00:00
3,2022-02-17 14:05:00+00:00
4,2022-02-17 14:10:00+00:00


## 15. Display Combined Data Preview
### Step Purpose: Verify Combined Dataset
Shows first few rows of the combined dataset to confirm successful merging.

**Why This Process/Method Was Needed:**
- **Reasoning**: After combining multiple files, need to verify the merge worked correctly
- **How It Helps Analysis**: Confirms patient_id column is present and data structure is intact
- **Analysis Impact If Skipped**: Might miss merge errors that corrupt the combined dataset

In [20]:
combined_df["time"] = pd.to_datetime(
    combined_df["time"],
    errors="coerce",
    utc=True
)

combined_df.to_csv(
    output_folder / "combined_cleaned_data.csv",
    index=False
)

print("Combined dataset updated with UTC time.")

Combined dataset updated with UTC time.


## 16. Save Combined Dataset
### Step Purpose: Persist Combined Results
Saves the final combined dataset with patient identifiers to CSV.

**Why This Process/Method Was Needed:**
- **Reasoning**: Combined dataset is the final deliverable for comprehensive analysis
- **How It Helps Analysis**: Enables loading the complete dataset into other analysis tools
- **Analysis Impact If Skipped**: Combined data only exists in memory, can't be reused or shared

## 17. Final Preview
### Step Purpose: Verify Combined Results
Displays final combined dataset to confirm successful processing.

**Why This Process/Method Was Needed:**
- **Reasoning**: Visual confirmation that all cleaning and combination steps worked correctly
- **How It Helps Analysis**: Allows manual inspection of final data before analysis
- **Analysis Impact If Skipped**: No direct impact, but reduces confidence in data quality